# Retrieval and Ranking

**Stage 1** — Bi-encoder dense retrieval via FAISS
- Top-5 standalone result -> `retrieval_biencoder_top5.jsonl`
- Top-20 candidate pool -> `retrieval_biencoder_top20.jsonl` (input for Stage 2 only)

**Stage 2** — Cross-encoder reranking (reads top-20, outputs top-5)
- Reranked result -> `retrieval_reranked_top5.jsonl`

In [19]:
!pip install sentence-transformers

In [2]:
import json
import os
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict

In [3]:
# Config
BENCHMARK_PATH = "../data/benchmark.json"
EMBEDDED_PATH = "../embedding/output/chunks_fixed300_overlap0/embedded_minilm.jsonl"
FAISS_PATH = "../embedding/output/chunks_fixed300_overlap0/embedded_minilm.faiss"
MODEL_NAME = "all-MiniLM-L6-v2"

RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
# Load benchmark
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)
print(f"Loaded {len(benchmark)} benchmark queries")

# Load FAISS index + corpus lines
index = faiss.read_index(FAISS_PATH)
with open(EMBEDDED_PATH, "r", encoding="utf-8") as f:
    corpus_lines = f.readlines()
assert len(corpus_lines) == index.ntotal
print(f"FAISS index: {index.ntotal} vectors, dim={index.d}")

# Load bi-encoder
bi_encoder = SentenceTransformer(MODEL_NAME)
print(f"Bi-encoder: {MODEL_NAME}")

Loaded 10 benchmark queries
FAISS index: 1838 vectors, dim=384


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder: all-MiniLM-L6-v2


In [5]:
def retrieve_biencoder(
    queries: List[Dict],
    index: faiss.Index,
    corpus_lines: List[str],
    model: SentenceTransformer,
    top_n: int,
) -> List[Dict]:
    """Bi-encoder retrieval for all queries. Returns top-n candidates per query."""
    query_texts = [q["query"] for q in queries]
    query_embs = model.encode(
        query_texts, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)

    scores_all, indices_all = index.search(query_embs, top_n)

    results = []
    for q, scores, indices in zip(queries, scores_all, indices_all):
        retrieved = []
        for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
            if idx == -1:
                continue
            chunk = json.loads(corpus_lines[idx])
            retrieved.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "score": round(float(score), 6),
                "text": chunk["text"],
            })
        results.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "retrieved": retrieved,
        })
    return results


def save_results(results: List[Dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} queries to {path}")

---

## Stage 1A: Top-5 (standalone)

Final retrieval result when no reranker is used. Ready for prompting and evaluation.

In [6]:
t0 = time.time()
results_top5 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, top_n=5)
print(f"Done: {len(results_top5)} queries, {time.time()-t0:.2f}s")

save_results(results_top5, os.path.join(RESULTS_DIR, "retrieval_biencoder_top5.jsonl"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Done: 10 queries, 0.42s
Saved 10 queries to results/retrieval_biencoder_top5.jsonl


In [7]:
# Inspect first query
sample = results_top5[0]
print(f"Query: {sample['query']}\n")
for r in sample["retrieved"]:
    print(f"  #{r['rank']}  score={r['score']:.4f}  {r['chunk_id']}")
    print(f"       {r['text'][:100]}...\n")

Query: What is sushi made of?

  #1  score=0.7450  wiki_0259_chunk_0
       Sushi (すし, 寿司, 鮨, 鮓; pronounced [sɯɕiꜜ] or [sɯꜜɕi] ) is a traditional Japanese dish made with vinega...

  #2  score=0.6993  wiki_0259_chunk_25
       of shellfish (abalone, prawn, scallop) are the most popular seafoods in sushi. Oysters are less comm...

  #3  score=0.6874  wiki_0259_chunk_27
       asa veitchii (熊笹, kuma-zasa) plants, respectively. Using actual leaves had the added benefit of rele...

  #4  score=0.6704  wiki_0259_chunk_7
       some other flavoring". There is an earlier mention of sushi in James Hepburn's Japanese–English dict...

  #5  score=0.6605  wiki_0259_chunk_22
       酢飯), shari (舎利), or gohan (ご飯)) is a preparation of white, short-grained, Japanese rice mixed with a...



---

## Stage 1B: Top-20 (for Stage 2 reranker)


It is a wider candidate pool so the cross-encoder reranker can pick better top-5 from 20 candidates instead of just reordering 5.

In [8]:
t0 = time.time()
results_top20 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, top_n=20)
print(f"Done: {len(results_top20)} queries, {time.time()-t0:.2f}s")

save_results(results_top20, os.path.join(RESULTS_DIR, "retrieval_biencoder_top20.jsonl"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Done: 10 queries, 0.14s
Saved 10 queries to results/retrieval_biencoder_top20.jsonl


---
---

## Stage 2: Cross-Encoder Reranking

Read the top-20 candidates above, rerank with a cross-encoder, output top-5.


In [11]:
from sentence_transformers import CrossEncoder

In [12]:
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP20_PATH = os.path.join(RESULTS_DIR, "retrieval_biencoder_top20.jsonl")
RERANKED_PATH = os.path.join(RESULTS_DIR, "retrieval_reranked_top5.jsonl")
TOP_K = 5

In [13]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f"Cross-encoder loaded: {CROSS_ENCODER_MODEL}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [20]:
with open(TOP20_PATH, "r", encoding="utf-8") as f:
    results_top20 = [json.loads(line) for line in f]
print(f"Loaded {len(results_top20)} queries from {TOP20_PATH}")

Loaded 10 queries from results/retrieval_biencoder_top20.jsonl


In [21]:
def rerank(
    stage1_results: List[Dict],
    cross_encoder: CrossEncoder,
    top_k: int = TOP_K,
) -> List[Dict]:
    """
    Re-rank Stage 1 candidates with a Cross-Encoder.

    For each query:
      1. Build (query, chunk_text) pairs for all top-20 candidates.
      2. Score every pair in one batch through the Cross-Encoder.
      3. Sort by Cross-Encoder score (descending).
      4. Return the top-k results with updated ranks.

    The original bi-encoder score is preserved as `score_biencoder`
    and the Cross-Encoder logit is stored as `score`.
    """
    reranked = []
    for item in stage1_results:
        query = item["query"]
        candidates = item["retrieved"]

        if not candidates:
            reranked.append(item)
            continue

        pairs = [(query, c["text"]) for c in candidates]
        ce_scores = cross_encoder.predict(pairs)

        scored = sorted(
            zip(ce_scores, candidates),
            key=lambda x: x[0],
            reverse=True,
        )

        top_k_hits = []
        for new_rank, (ce_score, cand) in enumerate(scored[:top_k], start=1):
            top_k_hits.append({
                "rank":            new_rank,
                "chunk_id":        cand["chunk_id"],
                "score":           cand["score"],
                "rerank_score":    round(float(ce_score), 6),
                "text":            cand["text"],
            })

        reranked.append({
            "query_id":  item["query_id"],
            "query":     query,
            "retrieved": top_k_hits,
        })
    return reranked

In [22]:
t0 = time.time()
results_reranked = rerank(results_top20, cross_encoder)
print(f"Done: {len(results_reranked)} queries, {time.time()-t0:.2f}s")

save_results(results_reranked, RERANKED_PATH)

Done: 10 queries, 9.13s
Saved 10 queries to results/retrieval_reranked_top5.jsonl


In [24]:
sample = results_reranked[0]
print(f"Query: {sample['query']}\n")
for r in sample["retrieved"]:
    print(f"  #{r['rank']}  ce={r['score']:.4f}  bi={r['rerank_score']:.4f}  {r['chunk_id']}")
    print(f"       {r['text'][:100]}...\n")

Query: What is sushi made of?

  #1  ce=0.7450  bi=9.1648  wiki_0259_chunk_0
       Sushi (すし, 寿司, 鮨, 鮓; pronounced [sɯɕiꜜ] or [sɯꜜɕi] ) is a traditional Japanese dish made with vinega...

  #2  ce=0.6485  bi=6.3095  wiki_0259_chunk_21
       crab or imitation crab cake topped with sushi ingredients. Maki poutine is similar to makizushi in s...

  #3  ce=0.6366  bi=5.4633  wiki_0259_chunk_1
       with pickled ginger (gari), wasabi, and soy sauce. Daikon radish or pickled daikon (takuan) are popu...

  #4  ce=0.6318  bi=4.9992  wiki_0259_chunk_14
       of sushi rice that has a strip of nori wrapped around its perimeter to form a vessel that is filled ...

  #5  ce=0.6137  bi=4.5674  wiki_0259_chunk_13
       ukemonoishi (pickling stone). As days pass, water seeps out and is removed. After six months, this s...



### Rank-change analysis

How much does the Cross-Encoder actually reorder the bi-encoder's top-5?

In [25]:
top1_changed = 0
rank_shifts   = []

for s1, s2 in zip(results_top5, results_reranked):
    s1_ids = [h["chunk_id"] for h in s1["retrieved"]]
    s2_ids = [h["chunk_id"] for h in s2["retrieved"]]

    if s1_ids and s2_ids and s1_ids[0] != s2_ids[0]:
        top1_changed += 1

    for new_rank, chunk_id in enumerate(s2_ids, start=1):
        if chunk_id in s1_ids:
            old_rank = s1_ids.index(chunk_id) + 1
            rank_shifts.append(abs(new_rank - old_rank))

n = len(results_top5)
avg_shift = sum(rank_shifts) / max(len(rank_shifts), 1)
print(f"Top-1 changed by reranking : {top1_changed}/{n} ({top1_changed/n*100:.0f}%)")
print(f"Avg absolute rank shift    : {avg_shift:.2f} positions")

Top-1 changed by reranking : 7/10 (70%)
Avg absolute rank shift    : 1.17 positions
